In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, time
import numpy as np
import pandas as pd
from astropy.timeseries import BoxLeastSquares
from scipy.optimize import minimize

INPUT = Path('/kaggle/input')
WORK = Path('/kaggle/working')
CASE_ROWS = np.array([4923, 5192, 27897, 27901, 40099, 40114, 63014, 63018, 86891, 87483, 89898, 89906], dtype=int)

roots = {}
for meta_path in INPUT.rglob('simulation_metadata.json'):
    if meta_path.parent.name != 'binary':
        continue
    try:
        meta = json.loads(meta_path.read_text())
    except Exception:
        continue
    if meta.get('family') == 'binary' and meta.get('parameter_draws_unique', 0) >= 100_000:
        roots[meta_path.parent.parent] = (meta_path.parent, meta)
if not roots:
    raise FileNotFoundError('Attach the Day 7 Cosmic swan Bayesian output containing simulations/binary/observations.npy and parameters.csv.')
SIM_ROOT, (BINARY_DIR, BINARY_META) = sorted(roots.items(), key=lambda item: item[1][1].get('created_utc',''))[-1]
for filename in ('observations.npy', 'parameters.csv', 'time_days.npy'):
    if not (BINARY_DIR / filename).is_file():
        raise FileNotFoundError(f'Missing binary simulation artifact: {BINARY_DIR / filename}')

binary_obs = np.load(BINARY_DIR / 'observations.npy', mmap_mode='r')
binary_params = pd.read_csv(BINARY_DIR / 'parameters.csv')
time_days = np.load(BINARY_DIR / 'time_days.npy').astype(float)
if binary_obs.shape[0] != len(binary_params) or binary_obs.shape[1] != len(time_days):
    raise ValueError('Binary observations, parameters, and time axis do not align.')
if CASE_ROWS.max() >= len(binary_params):
    raise ValueError('The Day 9.1 case indices do not exist in the attached simulation data.')

periods = binary_params['period_days'].to_numpy(float)
durations = binary_params['duration_fraction'].to_numpy(float)
depths = binary_params['depth_primary'].to_numpy(float)
noise = binary_params['noise_sigma'].to_numpy(float)
dt = float(np.median(np.diff(time_days)))
baseline = float(np.ptp(time_days))
eligible = ((periods*durations/dt >= 2.0) & (baseline*durations/dt >= 5.0)
            & (periods/dt >= 2.0) & (depths/noise >= 5.0))
if not eligible[CASE_ROWS].all():
    raise ValueError('At least one selected row does not meet the Day 9.1 observability criteria.')

OUT = WORK / ('day9-2-binary-map-diagnostics-' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ'))
OUT.mkdir(parents=True, exist_ok=False)
print('[PASS] Simulation source:', SIM_ROOT)
print('[PASS] Binary array shape:', binary_obs.shape)
print('[PASS] Reusing 12 Day 9.1 rows:', CASE_ROWS.tolist())
print('[PASS] Output directory:', OUT)
print('No GPU or NUTS sampling is used in this notebook.')

In [ ]:
def smooth_clip(x, sharpness=500.0):
    return (np.logaddexp(0.0, sharpness*x) - np.logaddexp(0.0, sharpness*(x-1.0))) / sharpness

def binary_prediction(t, period, phase, depth, ratio, duration, ellip, reflection):
    orbital = np.mod((t / period) - phase + 0.5, 1.0) - 0.5
    secondary_phase = np.mod(orbital, 1.0) - 0.5
    half_width = max(duration / 2.0, 1e-6)
    ingress = max(half_width * 0.18, 1e-6)
    primary = smooth_clip((half_width + ingress - np.abs(orbital)) / (2.0*ingress))
    secondary = smooth_clip((half_width + ingress - np.abs(secondary_phase)) / (2.0*ingress))
    return (1.0 - depth*primary - depth*ratio*secondary
            + ellip*np.cos(4.0*np.pi*orbital)
            + reflection*np.cos(2.0*np.pi*orbital))

def get_bls_start(t, y):
    bls = BoxLeastSquares(t, y)
    max_period = min(13.0, np.ptp(t)/2.0)
    bands = [(0.1,0.2),(0.2,0.4),(0.4,0.8),(0.8,1.6),(1.6,3.2),(3.2,6.4),(6.4,13.0)]
    trials = []
    for low, high in bands:
        high = min(high, max_period)
        if high <= low:
            continue
        max_duration = min(1.5, 0.95*low, 0.12*high)
        durations_days = np.geomspace(min(0.005, max_duration/2.0), max_duration, 16)
        result = bls.autopower(duration=durations_days, objective='snr', minimum_period=low,
                               maximum_period=high, frequency_factor=12.0)
        best = int(np.nanargmax(result.power))
        trials.append((float(result.power[best]), float(result.period[best]),
                       float(result.transit_time[best]), float(result.depth[best]),
                       float(result.duration[best])))
    if not trials:
        raise ValueError('BLS produced no valid period trials')
    power, period, transit, depth, duration = max(trials, key=lambda item: item[0])
    return {'power':power, 'period':period, 'phase':float(np.mod(transit/period,1.0)),
            'depth':float(np.clip(depth,0.002,0.45)),
            'duration_fraction':float(np.clip(duration/period,0.005,0.12))}

def fit_map(case_index):
    t = np.asarray(time_days, dtype=float)
    y = np.asarray(binary_obs[case_index], dtype=float)
    start = get_bls_start(t, y)
    period_low, period_high = max(0.1,start['period']-0.04), min(13.0,start['period']+0.04)
    phase0 = start['phase']
    bounds = [(period_low,period_high),(-0.08,0.08),(0.002,0.45),(0.0,1.0),
              (0.005,0.12),(0.0,0.02),(-0.01,0.01),(np.log(1e-4),np.log(0.03))]
    rough_noise = float(np.clip(np.std(np.diff(y))/np.sqrt(2.0),1e-4,0.03))
    theta0 = np.array([start['period'],0.0,start['depth'],0.5,start['duration_fraction'],
                       0.005,0.0,np.log(rough_noise)])
    theta0 = np.array([np.clip(v,lo+1e-8,hi-1e-8) for v,(lo,hi) in zip(theta0,bounds)])
    def objective(theta):
        period, phase_delta, depth, ratio, duration, ellip, reflection, log_noise = theta
        prediction = binary_prediction(t,period,np.mod(phase0+phase_delta,1.0),depth,ratio,duration,ellip,reflection)
        sigma = np.exp(log_noise)
        residual = y-prediction
        return float(0.5*np.sum((residual/sigma)**2 + 2.0*log_noise + np.log(2.0*np.pi)))
    result = minimize(objective,theta0,method='L-BFGS-B',bounds=bounds,
                      options={'maxiter':1600,'ftol':1e-10})
    theta = np.asarray(result.x,dtype=float)
    pred = binary_prediction(t,theta[0],np.mod(phase0+theta[1],1.0),theta[2],theta[3],theta[4],theta[5],theta[6])
    map_sigma = float(np.exp(theta[7]))
    names = ['period_days','phase_offset','depth_primary','depth_ratio','duration_fraction',
             'ellipsoidal_amplitude','reflection_amplitude','log_noise']
    values = dict(zip(names,theta.tolist()))
    truth = binary_params.iloc[case_index]
    truth_values = {
        'period_days':float(truth['period_days']),
        'phase_offset':float(((float(truth['phase'])-phase0+0.5)%1.0)-0.5),
        'depth_primary':float(truth['depth_primary']), 'depth_ratio':float(truth['depth_ratio']),
        'duration_fraction':float(truth['duration_fraction']),
        'ellipsoidal_amplitude':float(truth['ellipsoidal_amplitude']),
        'reflection_amplitude':float(truth['reflection_amplitude']),
        'log_noise':float(np.log(float(truth['noise_sigma']))),
    }
    boundary_hits=[]
    for name,value,(lo,hi) in zip(names,theta,bounds):
        if min(value-lo,hi-value)/(hi-lo) < 0.005:
            boundary_hits.append(name)
    period_in_window = bool(period_low <= truth_values['period_days'] <= period_high)
    case_row = {
        'simulation_row':int(case_index),'bls_period_days':start['period'],'bls_phase':phase0,
        'bls_power':start['power'],'known_period_days':truth_values['period_days'],
        'known_period_inside_bls_local_window':period_in_window,
        'local_period_window_low':period_low,'local_period_window_high':period_high,
        'map_success':bool(result.success),'map_message':str(result.message),
        'map_objective':float(result.fun),'map_noise_sigma':map_sigma,
        'rms_residual':float(np.sqrt(np.mean((y-pred)**2))),
        'rms_residual_over_map_noise':float(np.sqrt(np.mean((y-pred)**2))/map_sigma),
        'map_parameters_near_bounds':'|'.join(boundary_hits),
        'map_iterations':int(result.nit),
    }
    parameter_rows=[]
    for name in names:
        estimate=values[name]
        target=truth_values[name]
        parameter_rows.append({'simulation_row':int(case_index),'parameter':name,
                               'map_estimate':estimate,'known_truth':target,
                               'estimate_minus_truth':estimate-target,
                               'absolute_error':abs(estimate-target)})
    return case_row,parameter_rows

case_diagnostics=[]
parameter_diagnostics=[]
for i,row in enumerate(CASE_ROWS, start=1):
    tic=time.time()
    print(f'Case {i}/{len(CASE_ROWS)}: row {row}')
    try:
        case_row,param_rows=fit_map(int(row))
        case_row['seconds']=round(time.time()-tic,1)
        case_diagnostics.append(case_row)
        parameter_diagnostics.extend(param_rows)
        print('  MAP success:',case_row['map_success'],
              '| BLS period:',round(case_row['bls_period_days'],5),
              '| truth in local window:',case_row['known_period_inside_bls_local_window'],
              '| bound hits:',case_row['map_parameters_near_bounds'] or 'none')
    except Exception as exc:
        case_diagnostics.append({'simulation_row':int(row),'map_success':False,'error':repr(exc),
                                 'seconds':round(time.time()-tic,1)})
        print('  Diagnostic failed:',repr(exc))

case_df=pd.DataFrame(case_diagnostics)
parameter_df=pd.DataFrame(parameter_diagnostics)
case_df.to_csv(OUT/'day9_2_binary_map_cases.csv',index=False)
parameter_df.to_csv(OUT/'day9_2_binary_map_parameter_recovery.csv',index=False)
print('\nCase diagnostics:')
print(case_df.to_string(index=False))

In [ ]:
if len(parameter_df):
    print('MAP parameter absolute errors by parameter (all selected cases):')
    print(parameter_df.groupby('parameter')['absolute_error'].agg(['count','median','mean','max']).to_string())
if len(case_df):
    print('\nKnown period falls inside local BLS window:')
    print(case_df[['simulation_row','bls_period_days','known_period_days','known_period_inside_bls_local_window','map_success','map_parameters_near_bounds']].to_string(index=False))
summary={
    'created_utc':datetime.now(timezone.utc).isoformat(),
    'simulation_root':str(SIM_ROOT),
    'cases':CASE_ROWS.tolist(),
    'map_success_count':int(case_df['map_success'].fillna(False).sum()) if 'map_success' in case_df else 0,
    'truth_period_inside_local_bls_window_count':int(case_df['known_period_inside_bls_local_window'].fillna(False).sum()) if 'known_period_inside_bls_local_window' in case_df else 0,
    'scope':'BLS and MAP diagnostics only; no NUTS sampling.',
    'interpretation':'Use period-window misses, optimizer boundary hits, and residual-to-noise ratios to identify cases needing model or parameterization changes before another MCMC run.',
    'outputs':['day9_2_binary_map_cases.csv','day9_2_binary_map_parameter_recovery.csv'],
}
(OUT/'day9_2_binary_map_diagnostics_summary.json').write_text(json.dumps(summary,indent=2))
print('\nSaved diagnostics under:',OUT)
print(json.dumps(summary,indent=2))